# CS521 HW3 - Problem 1: reconstructed experiment

Jinwu Hu (jinwuhu2)

**Protocol limitation:** The referenced Q1 training starter and original HW1 checkpoints were unavailable. This experiment uses the exact public HW1 architecture (width-64 PreActResNet-18, softplus1, no normalization) and the public Q3 CIFAR-10 subsets. It does not claim to reproduce unobserved starter settings. Confirm this reconstruction is acceptable before submission.

The natural model is trained for 20 epochs; three independent copies are then adversarially fine-tuned for 10 epochs at epsilon 2/255, 4/255 and 8/255. Evaluation uses the same 1,000 test images, five evaluation radii, zero-start FGSM, and 50-step PGD with two random restarts.

In [ ]:
%pip install -q "gdown==5.2.2"


In [ ]:
from pathlib import Path
import hashlib
import gdown
data = Path("/content/cs521_hw3/data")
data.mkdir(parents=True,exist_ok=True)
files = {
    "linf_base.pt": "17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes",
    "cifar10_train_10k.pt": "1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe",
    "cifar10_test_1k.pt": "1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA",
}
expected_sha256 = {
    "linf_base.pt": "ba1d32efdc1951d9b2c5a47a735e4a1701c9ae4a62e57ad6ace58af0d19405a8",
    "cifar10_train_10k.pt": "dac9de04f3bfc3fea36381fe62894b34f7b66f73de46f3ac2502e567f3816c00",
    "cifar10_test_1k.pt": "cb14f0bede10d8dd0ee36c1bc58f5d11e69d61ff9834918a6318b8f63411ce7a",
}
for name, file_id in files.items():
    if not (data/name).exists():
        gdown.download(id=file_id, output=str(data/name), quiet=False)
    actual = hashlib.sha256((data/name).read_bytes()).hexdigest()
    assert actual == expected_sha256[name], f"Unexpected data/checkpoint content: {name}"


## Embedded helper functions

The helper module is created in memory from embedded code; no external Python source file is read.

In [ ]:
import types
helper = types.ModuleType('cs521_helper')
helper_source = '"""CS521 HW3 Q3: completed scaffold and reproducible experiment runner.\nAdapted from the official nalinwadhwa02/CCS521FA26HW notebook.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nROOT = Path("/content/cs521_hw3")\nROOT.mkdir(parents=True,exist_ok=True)\nDATA_DIR = ROOT / "data"\nimport time\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\ndevice = torch.device(os.environ.get("CS521_DEVICE", "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")))\nprint("device:", device)\n\nNORMS = ("linf", "l2", "l1")\nEPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}\n\ndef load_data(path):\n    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.\n    x, y = torch.load(DATA_DIR / path, map_location="cpu", weights_only=True)\n    return x.float().div(255).to(device), y.to(device)\n\n\n# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.\ndef initialize_data():\n    global x_train, y_train, x_test, y_test\n    x_train, y_train = load_data("cifar10_train_10k.pt")\n    x_test, y_test = load_data("cifar10_test_1k.pt")\n    print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape), flush=True)\n\n\ndef augment(x):\n    # Random horizontal flip and random crop (padding 4) for each image.\n    n = x.shape[0]\n    flip = torch.rand(n, device=x.device) < 0.5\n    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)\n    padded = F.pad(x, (4, 4, 4, 4))\n    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)\n    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)\n    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()\n\n\ndef batches(x, y, batch_size, shuffle):\n    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)\n    for k in range(0, len(x), batch_size):\n        j = idx[k:k + batch_size]\n        yield x[j], y[j]\n\n# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].\nclass PreActBlock(nn.Module):\n    def __init__(self, in_planes, planes, stride):\n        super().__init__()\n        self.bn1 = nn.BatchNorm2d(in_planes)\n        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)\n        self.bn2 = nn.BatchNorm2d(planes)\n        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)\n        self.shortcut = None\n        if stride != 1 or in_planes != planes:\n            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)\n\n    def forward(self, x):\n        out = F.relu(self.bn1(x))\n        shortcut = self.shortcut(out) if self.shortcut is not None else x\n        out = self.conv1(out)\n        out = self.conv2(F.relu(self.bn2(out)))\n        return out + shortcut\n\n\nclass PreActResNet18(nn.Module):\n    def __init__(self, width=32, num_classes=10):\n        super().__init__()\n        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))\n        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))\n        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)\n        layers, in_planes = [], width\n        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:\n            layers.append(PreActBlock(in_planes, mult * width, stride))\n            in_planes = mult * width\n        self.layers = nn.Sequential(*layers)\n        self.bn = nn.BatchNorm2d(in_planes)\n        self.linear = nn.Linear(in_planes, num_classes)\n\n    def forward(self, x):\n        out = self.layers(self.conv1((x - self.mean) / self.std))\n        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)\n        return self.linear(out)\n\n\ndef make_model():\n    return PreActResNet18().to(device)\n\n\ndef load_base_model():\n    model = make_model()\n    model.load_state_dict(torch.load(DATA_DIR / "linf_base.pt", map_location=device, weights_only=True))\n    return model\n\ndef clip_box(x, delta):\n    # Make x + delta an image in [0, 1]. This never increases |delta_i|.\n    return (x + delta).clamp(0, 1) - x\n\n\ndef input_grad(model, x, delta, y):\n    delta = delta.detach().requires_grad_(True)\n    loss = F.cross_entropy(model(x + delta), y, reduction="sum")\n    (grad,) = torch.autograd.grad(loss, delta)\n    return grad.detach()\n\n\ndef l2_norm(v):\n    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)\n\n\ndef pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):\n    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)\n    delta = clip_box(x, delta)\n    for _ in range(steps):\n        grad = input_grad(model, x, delta, y)\n        delta = clip_box(x, (delta + alpha * grad.sign()).clamp(-eps, eps))\n    return (x + delta).detach()\n\ndef pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):\n    delta = torch.zeros_like(x)\n    if rand_init:\n        delta = torch.randn_like(x)\n        # Uniform radius, matching the random-radius convention of the supplied L1 attack.\n        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)\n        delta = delta / l2_norm(delta).clamp_min(1e-12) * (eps * r)\n    delta = clip_box(x, delta)\n    for _ in range(steps):\n        grad = input_grad(model, x, delta, y)\n        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)\n        delta = delta * (eps / l2_norm(delta).clamp_min(1e-12)).clamp(max=1.0)\n        delta = clip_box(x, delta)\n    return (x + delta).detach()\n\n\ndef project_l1(delta, eps):\n    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).\n    v = delta.flatten(1)\n    a = v.abs()\n    outside = a.sum(dim=1) > eps\n    if not outside.any():\n        return delta\n    mu, _ = a.sort(dim=1, descending=True)\n    cumsum = mu.cumsum(dim=1)\n    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)\n    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)\n    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)\n    w = (a - theta).clamp_min(0) * v.sign()\n    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)\n\n\ndef pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):\n    # Each step changes only the k coordinates with the largest gradient.\n    # Coordinates at the image boundary that point outward are skipped.\n    delta = torch.zeros_like(x)\n    if rand_init:\n        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)\n        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)\n        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r\n    delta = clip_box(x, delta)\n    for _ in range(steps):\n        grad = input_grad(model, x, delta, y).flatten(1)\n        x_cur = (x + delta).flatten(1)\n        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))\n        grad = grad.masked_fill(blocked, 0)\n        top = grad.abs().topk(k, dim=1).indices\n        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())\n        delta = clip_box(x, delta + alpha * (step / k).view_as(x))\n        delta = project_l1(delta, eps)\n    return (x + delta).detach()\n\n\nATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}\n\n\ndef attack(model, x, y, norm, steps=10):\n    # The attack for one norm with the default eps and step size.\n    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps\n    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)\n\ndef test_model(model, steps=50, restarts=2, batch_size=250):\n    model.eval()\n    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable\n    start = time.time()\n    correct = {k: 0 for k in ("clean", *NORMS, "union")}\n    for i in range(0, len(x_test), batch_size):\n        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]\n        with torch.no_grad():\n            clean = model(x).argmax(1) == y\n        union = clean.clone()\n        for norm in NORMS:\n            robust = clean.clone()\n            for _ in range(restarts):\n                x_adv = attack(model, x, y, norm, steps)\n                with torch.no_grad():\n                    robust &= model(x_adv).argmax(1) == y\n            correct[norm] += robust.sum().item()\n            union &= robust\n        correct["clean"] += clean.sum().item()\n        correct["union"] += union.sum().item()\n    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}\n    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")\n    return acc\n\ndef adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):\n    torch.manual_seed(seed)\n    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))\n    params = list(model.parameters())\n    start = time.time()\n    history = []\n    for epoch in range(epochs):\n        sync_device()\n        epoch_start = time.perf_counter()\n        losses = []\n        for x, y in batches(x_train, y_train, batch_size, shuffle=True):\n            if len(y) < batch_size:  # skip the last, smaller batch\n                continue\n            x = augment(x)\n            loss = loss_fn(model, x, y, **kwargs)\n            opt.zero_grad()\n            if gp_beta is None:\n                loss.backward()\n            else:\n                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n\n                # for each parameter tensor. gradient_projection is defined in part (b).\n                g_a = torch.autograd.grad(loss, params)\n                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)\n                for p, ga, gn in zip(params, g_a, g_n):\n                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga\n            opt.step()\n            sched.step()\n            losses.append(loss.item())\n        sync_device()\n        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)), "seconds": time.perf_counter() - epoch_start})\n        print(json.dumps(history[-1]), flush=True)\n    model.eval()\n    model.training_history = history\n    return model\n\ndef single_norm_loss(model, x, y, norm):\n    model.eval()\n    x_adv = attack(model, x, y, norm)\n    model.train()\n    return F.cross_entropy(model(x_adv), y)\n\n\ndef ramp_loss(model, x, y, lam=0.5):\n    model.eval()\n    x_q = attack(model, x, y, "linf")\n    x_r = attack(model, x, y, "l1")\n    model.train()\n    logits_q, logits_r = model(x_q), model(x_r)\n    ce_q = F.cross_entropy(logits_q, y, reduction="none")\n    ce_r = F.cross_entropy(logits_r, y, reduction="none")\n    # PyTorch\'s target is p_q, input is log p_r: this is KL(p_q || p_r).\n    kl = F.kl_div(F.log_softmax(logits_r, dim=1), F.softmax(logits_q, dim=1),\n                  reduction="none").sum(dim=1)\n    mask = (logits_q.argmax(1) == y).to(kl.dtype)\n    # Mean over the qualifying examples; exactly zero when none qualify.\n    paired = (kl * mask).sum() / mask.sum().clamp_min(1)\n    return torch.maximum(ce_q, ce_r).mean() + lam * paired\n\ndef gradient_projection(g_n, g_a):\n    cosine = (g_n.flatten() @ g_a.flatten()) / (g_n.norm() * g_a.norm()).clamp_min(1e-12)\n    return cosine.clamp_min(0) * g_n\n\n\nclass AlternatingDistillationLoss:\n    """Alternate L1/L2 updates while retaining an L-infinity-robust teacher.\n\n    This is a student-method candidate, not a claim of global research novelty.\n    Teacher targets never depend on test examples or test labels.\n    """\n    def __init__(self, teacher, distill_weight=0.5, clean_weight=0.25):\n        self.teacher = teacher.eval()\n        for p in self.teacher.parameters():\n            p.requires_grad_(False)\n        self.distill_weight = distill_weight\n        self.clean_weight = clean_weight\n        self.batch_index = 0\n        self.stats = []\n\n    def __call__(self, model, x, y):\n        norm = ("l1", "l2")[self.batch_index % 2]\n        self.batch_index += 1\n        model.eval()\n        x_alt = attack(model, x, y, norm)\n        # Keep attack cost identical in the ablation to isolate distillation.\n        x_anchor = attack(model, x, y, "linf")\n        with torch.no_grad():\n            teacher_logits = self.teacher(x)\n            teacher_prob = F.softmax(teacher_logits, dim=1)\n            reliable = (teacher_logits.argmax(1) == y)\n        model.train()\n        alt_logits = model(x_alt)\n        anchor_logits = model(x_anchor)\n        clean_logits = model(x)\n        ce_alt = F.cross_entropy(alt_logits, y)\n        ce_clean = F.cross_entropy(clean_logits, y)\n        kd_each = F.kl_div(F.log_softmax(anchor_logits, dim=1), teacher_prob,\n                          reduction="none").sum(dim=1)\n        mask = reliable.to(kd_each.dtype)\n        kd = (kd_each * mask).sum() / mask.sum().clamp_min(1)\n        loss = ce_alt + self.distill_weight * kd + self.clean_weight * ce_clean\n        self.stats.append({"norm": norm, "ce_alt": ce_alt.detach().item(),\n                           "clean_ce": ce_clean.detach().item(), "distillation": kd.detach().item(),\n                           "teacher_correct_fraction": mask.mean().item()})\n        return loss\n\n_my_loss_instance = None\ndef my_loss(model, x, y):\n    global _my_loss_instance\n    if _my_loss_instance is None:\n        _my_loss_instance = AlternatingDistillationLoss(load_base_model())\n    return _my_loss_instance(model, x, y)\n\n\n\ndef sync_device():\n    if device.type == "cuda": torch.cuda.synchronize()\n    elif device.type == "mps": torch.mps.synchronize()\n\ndef save_model(model, path):\n    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, path)\n\ndef execute_run(method, seed, epochs=10):\n    dest = ROOT / "results/q3" / f"{method}_seed{seed}_epochs{epochs}"\n    dest.mkdir(parents=True, exist_ok=True)\n    result_path = dest / "result.json"\n    if result_path.exists():\n        print("SKIP_COMPLETE", dest.name, flush=True)\n        return json.loads(result_path.read_text())\n    torch.manual_seed(seed)\n    model = load_base_model()\n    loss_fn = None\n    extra = {}\n    if method in NORMS:\n        loss_fn = single_norm_loss\n        extra["norm"] = method\n    elif method == "ramp":\n        loss_fn = ramp_loss\n        extra["gp_beta"] = 0.5\n    elif method in ["alternating_distillation", "without_distillation"]:\n        loss_fn = AlternatingDistillationLoss(load_base_model(),\n                    distill_weight=0.5 if method == "alternating_distillation" else 0.0)\n    elif method != "base":\n        raise ValueError(method)\n    if loss_fn is not None:\n        model = adv_train(model, loss_fn, epochs=epochs, seed=seed, **extra)\n        save_model(model, dest / "model.pt")\n        (dest / "training_history.json").write_text(json.dumps(model.training_history, indent=2) + "\\n")\n        if isinstance(loss_fn, AlternatingDistillationLoss):\n            (dest / "batch_statistics.json").write_text(json.dumps(loss_fn.stats) + "\\n")\n    sync_device()\n    start = time.perf_counter()\n    acc = test_model(model)  # Required 1,000 examples, 50 steps, two restarts.\n    sync_device()\n    result = {"method": method, "seed": seed, "epochs": epochs if method != "base" else 0,\n              "accuracy_percent": acc, "evaluation_seconds": time.perf_counter() - start,\n              "training_history": getattr(model, "training_history", []),\n              "evaluation": {"n": 1000, "steps": 50, "restarts": 2, "batch_size": 250},\n              "device": str(device), "torch": torch.__version__,\n              "canonical_module_sha256": \'8160d9711962fae02733f82c85013c2446649727199b37a1e74f733a9e300035\'}\n    result_path.write_text(json.dumps(result, indent=2) + "\\n")\n    print("RUN_COMPLETE", method, seed, json.dumps(acc), flush=True)\n    del model, loss_fn\n    if device.type == "cuda": torch.cuda.empty_cache()\n    elif device.type == "mps": torch.mps.empty_cache()\n    return result\n\ndef benchmark():\n    model = load_base_model()\n    model.eval()\n    x, y = x_train[:128], y_train[:128]\n    for norm in NORMS:\n        sync_device()\n        start = time.perf_counter()\n        x_adv = attack(model, x, y, norm, steps=10)\n        sync_device()\n        print("BENCHMARK", norm, time.perf_counter() - start, flush=True)\n        delta = x_adv - x\n        ord_value = {"linf": float("inf"), "l2": 2, "l1": 1}[norm]\n        max_norm = delta.flatten(1).norm(p=ord_value, dim=1).max().item()\n        assert max_norm <= EPS[norm] + 1e-4, (norm, max_norm)\n        assert x_adv.min() >= 0 and x_adv.max() <= 1\n        assert not model.training\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--method", default="benchmark", choices=["benchmark","base",*NORMS,"ramp","alternating_distillation","without_distillation","all","strong"])\n    p.add_argument("--seed", type=int, default=0)\n    p.add_argument("--epochs", type=int, default=10)\n    p.add_argument("--checkpoint")\n    a = p.parse_args()\n    torch.set_num_threads(4)\n    initialize_data()\n    if a.method == "benchmark":\n        benchmark()\n    elif a.method == "all":\n        schedule = [("base",0),("linf",0),("l2",0),("l1",0),("ramp",0),("ramp",1),\n                    ("alternating_distillation",0),("alternating_distillation",1),\n                    ("without_distillation",0),("without_distillation",1)]\n        for method, seed in schedule: execute_run(method, seed)\n    elif a.method == "strong":\n        if not a.checkpoint: raise ValueError("--checkpoint is required")\n        model = make_model()\n        model.load_state_dict(torch.load(a.checkpoint, map_location=device, weights_only=True))\n        acc = test_model(model, steps=200, restarts=5)\n        Path(a.checkpoint).with_name("strong_evaluation.json").write_text(\n            json.dumps({"accuracy_percent": acc, "steps":200, "restarts":5, "n":1000}, indent=2) + "\\n")\n    else:\n        execute_run(a.method, a.seed, a.epochs)\n\n'
exec(helper_source, helper.__dict__)


## Official HW1 model definitions

In [ ]:
"""Model definitions copied from the official HW1 public notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F

def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2

def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2

class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std

class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs
    
class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
        activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512*block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            # layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
    normal='none'):
    #print('initializing PA RN-18 with act {}, normal {}'.format())
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
        activation=activation, fts_before_bn=fts_before_bn, normal=normal)




## Training and evaluation

In [ ]:
"""Q1 reconstructed experiment using the public HW1 architecture.

The original Q1 training notebook / HW1 checkpoints were unavailable.
This explicitly documented protocol is an independent reconstruction, not a
claim that the missing starter's hyperparameters have been recovered.
"""
import argparse
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F




ROOT = Path("/content/cs521_hw3")
DEST = ROOT / "results/q1"
device = helper.device

def new_model():
    return PreActResNet18(10, cuda=device.type == "cuda",
                         activation="softplus1", normal="none").to(device)

def train(model, eps, epochs, lr, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, epochs * (len(helper.x_train) // 128))
    history = []
    for epoch in range(epochs):
        helper.sync_device()
        before = time.perf_counter()
        losses = []
        for x, y in helper.batches(helper.x_train, helper.y_train, 128, True):
            if len(y) < 128:
                continue
            x = helper.augment(x)
            if eps:
                model.eval()
                x = helper.pgd_linf(model, x, y, eps, 2.5 * eps / 10, 10, True)
            model.train()
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        helper.sync_device()
        history.append({"epoch":epoch+1, "mean_loss":float(np.mean(losses)),
                        "seconds":time.perf_counter()-before})
        print("Q1_TRAIN", eps, json.dumps(history[-1]), flush=True)
    return history

def evaluate(model):
    model.eval()
    rows = []
    for units in [0, 2, 4, 8, 12]:
        eps = units / 255
        torch.manual_seed(42)
        counts = {"clean":0,"fgsm":0,"pgd":0}
        helper.sync_device()
        before = time.perf_counter()
        for x, y in helper.batches(helper.x_test, helper.y_test, 125, False):
            with torch.no_grad():
                correct = model(x).argmax(1) == y
            counts["clean"] += correct.sum().item()
            if units == 0:
                counts["fgsm"] += correct.sum().item()
                counts["pgd"] += correct.sum().item()
                continue
            fgsm = helper.pgd_linf(model, x, y, eps, eps, 1, False)
            with torch.no_grad():
                counts["fgsm"] += (correct & (model(fgsm).argmax(1) == y)).sum().item()
            robust = correct.clone()
            for _ in range(2):
                adv = helper.pgd_linf(model, x, y, eps, eps / 4, 50, True)
                with torch.no_grad():
                    robust &= model(adv).argmax(1) == y
            counts["pgd"] += robust.sum().item()
        helper.sync_device()
        row = {"test_eps_over_255":units,
               "accuracy_percent":{k:100*v/len(helper.y_test) for k,v in counts.items()},
               "seconds":time.perf_counter()-before}
        rows.append(row)
        print("Q1_EVALUATE",json.dumps(row),flush=True)
    return rows

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--mode",choices=["all","clean","adv"],default="all")
    p.add_argument("--eps",type=int,default=4)
    args = p.parse_args()
    torch.set_num_threads(4)
    helper.initialize_data()
    DEST.mkdir(parents=True,exist_ok=True)
    protocol = {
        "status":"Reconstructed protocol; missing original Q1 training starter",
        "architecture":"Public HW1 PreActResNet18, width 64, softplus1, no input normalization",
        "train":"Official Q3 balanced 10000-image CIFAR-10 training subset",
        "test":"Official Q3 first 1000 CIFAR-10 test images; no training on test",
        "clean_epochs":20,"clean_lr":.05,"adversarial_epochs":10,"adversarial_lr":.01,
        "training_eps_over_255":[2,4,8],"batch_size":128,"seed":0,
        "optimizer":"SGD(momentum=.9,weight_decay=.0005), cosine LR",
        "augmentation":"random horizontal flip and random crop, padding 4",
        "pgd_train":{"steps":10,"alpha":"2.5*eps/10","random_start":True},
        "pgd_eval":{"steps":50,"restarts":2,"alpha":"eps/4","random_start":True},
        "fgsm_eval":{"steps":1,"alpha":"eps","random_start":False},
        "canonical_module_sha256":'a53c6b7e7e71823bef85bf2ff7b26fc6769e60071741315deb33f621e7d1dd21',
    }
    (DEST/"protocol.json").write_text(json.dumps(protocol,indent=2))
    original = DEST/"original_clean"
    original.mkdir(exist_ok=True)
    if args.mode in ["all","clean"] and not (original/"result.json").exists():
        torch.manual_seed(0)
        model = new_model()
        if not (original/"model.pt").exists():
            history = train(model, 0, 20, .05)
            helper.save_model(model, original/"model.pt")
            (original/"training_history.json").write_text(json.dumps(history,indent=2))
        else:
            model.load_state_dict(torch.load(original/"model.pt",weights_only=True,map_location=device))
        results = evaluate(model)
        (original/"result.json").write_text(json.dumps(results,indent=2))
        del model
    eps_values = [2,4,8] if args.mode == "all" else [args.eps] if args.mode == "adv" else []
    for units in eps_values:
        dest = DEST/f"pgd_train_eps{units}_over255"
        dest.mkdir(exist_ok=True)
        if (dest/"result.json").exists():
            continue
        model = new_model()
        if not (dest/"model.pt").exists():
            model.load_state_dict(torch.load(original/"model.pt",weights_only=True,map_location=device))
            history = train(model, units/255, 10, .01)
            helper.save_model(model,dest/"model.pt")
            (dest/"training_history.json").write_text(json.dumps(history,indent=2))
        else:
            model.load_state_dict(torch.load(dest/"model.pt",weights_only=True,map_location=device))
        results = evaluate(model)
        (dest/"result.json").write_text(json.dumps(results,indent=2))
        del model
    print("Q1_COMPLETE",flush=True)



In [ ]:
import sys
sys.argv = ["q1", "--mode", "all"]
main()


## Recorded experiment evidence

The following saved measurements came from the completed CUDA experiment. They document the explicitly reconstructed protocol above, not the unavailable original training settings.

In [ ]:
recorded_evaluations = {'original_clean': [{'test_eps_over_255': 0, 'accuracy_percent': {'clean': 73.5, 'fgsm': 73.5, 'pgd': 73.5}, 'seconds': 0.07699334000062663}, {'test_eps_over_255': 2, 'accuracy_percent': {'clean': 73.5, 'fgsm': 24.8, 'pgd': 15.5}, 'seconds': 8.014916535998054}, {'test_eps_over_255': 4, 'accuracy_percent': {'clean': 73.5, 'fgsm': 8.2, 'pgd': 0.5}, 'seconds': 7.998185301999911}, {'test_eps_over_255': 8, 'accuracy_percent': {'clean': 73.5, 'fgsm': 3.9, 'pgd': 0.0}, 'seconds': 7.677562704997399}, {'test_eps_over_255': 12, 'accuracy_percent': {'clean': 73.5, 'fgsm': 2.8, 'pgd': 0.0}, 'seconds': 7.644408323998505}], 'pgd_train_eps2_over255': [{'test_eps_over_255': 0, 'accuracy_percent': {'clean': 71.3, 'fgsm': 71.3, 'pgd': 71.3}, 'seconds': 0.03839523100032238}, {'test_eps_over_255': 2, 'accuracy_percent': {'clean': 71.3, 'fgsm': 56.8, 'pgd': 55.9}, 'seconds': 8.109250839999731}, {'test_eps_over_255': 4, 'accuracy_percent': {'clean': 71.3, 'fgsm': 39.9, 'pgd': 37.7}, 'seconds': 8.040843334001693}, {'test_eps_over_255': 8, 'accuracy_percent': {'clean': 71.3, 'fgsm': 15.6, 'pgd': 8.2}, 'seconds': 8.030472765000013}, {'test_eps_over_255': 12, 'accuracy_percent': {'clean': 71.3, 'fgsm': 5.7, 'pgd': 0.8}, 'seconds': 8.018629230999068}], 'pgd_train_eps4_over255': [{'test_eps_over_255': 0, 'accuracy_percent': {'clean': 64.5, 'fgsm': 64.5, 'pgd': 64.5}, 'seconds': 0.03619153399995412}, {'test_eps_over_255': 2, 'accuracy_percent': {'clean': 64.5, 'fgsm': 54.7, 'pgd': 54.5}, 'seconds': 7.817998955997609}, {'test_eps_over_255': 4, 'accuracy_percent': {'clean': 64.5, 'fgsm': 45.0, 'pgd': 44.3}, 'seconds': 7.738954292999551}, {'test_eps_over_255': 8, 'accuracy_percent': {'clean': 64.5, 'fgsm': 26.0, 'pgd': 22.4}, 'seconds': 7.659459656999388}, {'test_eps_over_255': 12, 'accuracy_percent': {'clean': 64.5, 'fgsm': 13.9, 'pgd': 6.6}, 'seconds': 7.6584814169982565}], 'pgd_train_eps8_over255': [{'test_eps_over_255': 0, 'accuracy_percent': {'clean': 53.9, 'fgsm': 53.9, 'pgd': 53.9}, 'seconds': 0.03552107300129137}, {'test_eps_over_255': 2, 'accuracy_percent': {'clean': 53.9, 'fgsm': 48.8, 'pgd': 48.7}, 'seconds': 7.822493863997806}, {'test_eps_over_255': 4, 'accuracy_percent': {'clean': 53.9, 'fgsm': 43.8, 'pgd': 43.9}, 'seconds': 7.807732709999982}, {'test_eps_over_255': 8, 'accuracy_percent': {'clean': 53.9, 'fgsm': 31.4, 'pgd': 30.8}, 'seconds': 7.821705840000504}, {'test_eps_over_255': 12, 'accuracy_percent': {'clean': 53.9, 'fgsm': 22.6, 'pgd': 20.8}, 'seconds': 8.040024338999501}]}
print(json.dumps(recorded_evaluations, indent=2))
